# 당뇨 분류 실습 (원핫인코딩 버전)

ex_0914 실습을 원핫인코딩으로 바꾼 노트북입니다. 모델 5개(의사결정나무, 랜덤포레스트, 로지스틱 회귀, KNN, 의사결정나무 회귀)와 1~5단계 구성은 그대로 두고, 아래 세 가지만 달라졌습니다.

1. **0을 미측정으로 처리합니다.** Glucose, BloodPressure, SkinThickness, Insulin, BMI의 0은 실제 값이 아니라 측정이 빠진 값입니다.
2. **숫자를 구간으로 나눈 뒤 원핫인코딩합니다.** 이 데이터는 8개 column이 모두 숫자라서 그대로는 원핫인코딩할 것이 없습니다. 그래서 나이를 20대, 30대처럼 구간(범주)으로 바꾼 뒤 원핫인코딩합니다.
3. **공통 단계는 한 번만 실행합니다.** 1~2단계(불러오기, 전처리, 분할)를 먼저 실행하고, 모델마다 3~5단계만 실행합니다.

위에서부터 차례로 실행하세요.

In [1]:
# [0단계] 코랩 준비: 구글 드라이브 연결
import os

DRIVE_PATH = "/content/drive/MyDrive/Colab Notebooks/diabetes.csv"   # 원래 실습에서 쓰던 경로

try:
    from google.colab import drive
    drive.mount('/content/drive')    # 처음 실행하면 구글 계정 연결 창이 뜹니다
except ImportError:
    pass                             # 코랩이 아니면 건너뜁니다

## [1단계] 데이터 불러오기
드라이브 경로에 파일이 없으면 현재 폴더의 `diabetes.csv`, `diabetes_1.csv`를 찾고, 그래도 없으면 업로드 창을 띄웁니다.

In [2]:
# [1단계] 데이터 불러오기
import pandas as pd
import numpy as np

candidates = [DRIVE_PATH, "diabetes.csv", "diabetes_1.csv"]
CSV_PATH = next((p for p in candidates if os.path.exists(p)), None)

if CSV_PATH is None:
    from google.colab import files
    uploaded = files.upload()          # 파일 선택 창에서 csv 파일을 고릅니다
    CSV_PATH = next(iter(uploaded))

df = pd.read_csv(CSV_PATH)

# 독립 변수(X)와 타겟 변수(y) 분리
# 'Outcome' 컬럼이 당뇨 여부(정답)를 나타냅니다 (0 또는 1).
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

print("읽은 파일:", CSV_PATH)
print("데이터 형태 (행, 열):", df.shape)
print("상위 5개 데이터 미리보기:")
display(df.head())

읽은 파일: diabetes.csv
데이터 형태 (행, 열): (768, 9)
상위 5개 데이터 미리보기:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## [1-1단계] 0을 미측정(NaN)으로 바꾸기
혈당, 혈압, BMI가 0인 사람은 없습니다. 이 0은 측정이 빠진 값이라서 NaN(값 없음)으로 바꿉니다.
Pregnancies(임신 횟수)의 0은 실제 값이므로 바꾸지 않습니다.

In [3]:
# [1-1단계] 0을 미측정(NaN)으로 바꾸기
missing_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

print("column별 0의 개수:")
display((X[missing_cols] == 0).sum().to_frame("0의 개수"))

X_clean = X.copy()
X_clean[missing_cols] = X_clean[missing_cols].replace(0, np.nan)

column별 0의 개수:


,0의 개수
Glucose,5
BloodPressure,35
SkinThickness,227
Insulin,374
BMI,11


## [1-2단계] 숫자를 구간(범주)으로 나누기
원핫인코딩은 "남/여", "서울/부산"처럼 **범주**를 0과 1의 column으로 바꾸는 방법입니다. 숫자 column은 먼저 구간으로 나눠 범주로 만들어야 합니다.

| column | 구간 | 정한 근거 |
| --- | --- | --- |
| Glucose | 100 미만, 100-139, 140-169, 170 이상 | 포도당 부하 2시간 후 혈당. 140 미만이 정상 범위 |
| BloodPressure | 70 미만, 70-79, 80-89, 90 이상 | 이완기 혈압. 80과 90이 고혈압 단계 기준 |
| BMI | 25 미만, 25-29.9, 30-34.9, 35 이상 | WHO 기준 과체중 25, 비만 30 |
| Age | 20대, 30대, 40대, 50세 이상 | 연령대 |
| Pregnancies | 0회, 1-2회, 3-5회, 6회 이상 | 인원이 비슷하게 나뉘도록 |
| SkinThickness, Insulin, DiabetesPedigreeFunction | 끊기 좋은 값 | 실습용으로 임의로 정함 |

1-1단계에서 NaN으로 바꾼 값은 `미측정`이라는 구간으로 따로 둡니다.

In [4]:
# [1-2단계] 숫자를 구간(범주)으로 나누기
inf = np.inf
BINS = {   # column: (구간 경계, 구간 이름)   경계는 [이상, 미만)
    "Pregnancies":              ([0, 1, 3, 6, inf],       ["0회", "1-2회", "3-5회", "6회 이상"]),
    "Glucose":                  ([0, 100, 140, 170, inf], ["100 미만", "100-139", "140-169", "170 이상"]),
    "BloodPressure":            ([0, 70, 80, 90, inf],    ["70 미만", "70-79", "80-89", "90 이상"]),
    "SkinThickness":            ([0, 20, 30, 40, inf],    ["20 미만", "20-29", "30-39", "40 이상"]),
    "Insulin":                  ([0, 100, 200, inf],      ["100 미만", "100-199", "200 이상"]),
    "BMI":                      ([0, 25, 30, 35, inf],    ["25 미만", "25-29.9", "30-34.9", "35 이상"]),
    "DiabetesPedigreeFunction": ([0, 0.3, 0.6, inf],      ["0.3 미만", "0.3-0.59", "0.6 이상"]),
    "Age":                      ([0, 30, 40, 50, inf],    ["20대", "30대", "40대", "50세 이상"]),
}

X_cat = pd.DataFrame(index=X_clean.index)
for col, (bins, labels) in BINS.items():
    binned = pd.cut(X_clean[col], bins=bins, labels=labels, right=False)
    if col in missing_cols:
        binned = binned.cat.add_categories("미측정").fillna("미측정")
    X_cat[col] = binned

print("구간으로 바꾼 결과 (상위 5개):")
display(X_cat.head())

# 구간마다 인원과 당뇨 비율을 확인합니다
for col in ["Glucose", "BMI", "Age"]:
    table = df.groupby(X_cat[col], observed=False)["Outcome"].agg(인원="count", 당뇨_비율="mean").round(2)
    print(f"\n[{col}] 구간별 인원과 당뇨 비율")
    display(table)

구간으로 바꾼 결과 (상위 5개):


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6회 이상,140-169,70-79,30-39,미측정,30-34.9,0.6 이상,50세 이상
1,1-2회,100 미만,70 미만,20-29,미측정,25-29.9,0.3-0.59,30대
2,6회 이상,170 이상,70 미만,미측정,미측정,25 미만,0.6 이상,30대
3,1-2회,100 미만,70 미만,20-29,100 미만,25-29.9,0.3 미만,20대
4,0회,100-139,70 미만,30-39,100-199,35 이상,0.6 이상,30대



[Glucose] 구간별 인원과 당뇨 비율


,인원,당뇨_비율
Glucose,,
100 미만,192,0.07
100-139,374,0.31
140-169,126,0.60
170 이상,71,0.85
미측정,5,0.40



[BMI] 구간별 인원과 당뇨 비율


,인원,당뇨_비율
BMI,,
25 미만,106,0.07
25-29.9,179,0.22
30-34.9,224,0.45
35 이상,248,0.48
미측정,11,0.18



[Age] 구간별 인원과 당뇨 비율


,인원,당뇨_비율
Age,,
20대,396,0.21
30대,165,0.46
40대,118,0.55
50세 이상,89,0.48


## [1-3단계] 원핫인코딩
구간마다 column을 하나씩 만들고, 그 구간에 해당하면 1, 아니면 0을 넣습니다. 먼저 나이 5명으로 작게 확인합니다.

In [5]:
# [1-3단계] 원핫인코딩: 작은 예시
example = X_cat[["Age"]].head(5)
print("원핫인코딩 전:")
display(example)
print("원핫인코딩 후: Age 1개 column이 구간 수만큼 4개 column이 됩니다")
display(pd.get_dummies(example, dtype=int))

원핫인코딩 전:


,Age
0,50세 이상
1,30대
2,30대
3,20대
4,30대


원핫인코딩 후: Age 1개 column이 구간 수만큼 4개 column이 됩니다


,Age_20대,Age_30대,Age_40대,Age_50세 이상
0,0,0,0,1
1,0,1,0,0
2,0,1,0,0
3,1,0,0,0
4,0,1,0,0


In [6]:
# [1-3단계] 원핫인코딩: 전체 데이터
X_onehot = pd.get_dummies(X_cat, dtype=int)

print("원핫인코딩 전:", X_cat.shape, " -> 원핫인코딩 후:", X_onehot.shape)
print("모든 행의 1의 개수:", X_onehot.sum(axis=1).unique(), " (원래 column 8개마다 1이 하나씩)")

print("\n첫 번째 환자에게 1이 들어간 column:")
print(X_onehot.columns[X_onehot.iloc[0] == 1].tolist())

display(X_onehot.head(3))

원핫인코딩 전: (768, 8)  -> 원핫인코딩 후: (768, 35)
모든 행의 1의 개수: [8]  (원래 column 8개마다 1이 하나씩)

첫 번째 환자에게 1이 들어간 column:
['Pregnancies_6회 이상', 'Glucose_140-169', 'BloodPressure_70-79', 'SkinThickness_30-39', 'Insulin_미측정', 'BMI_30-34.9', 'DiabetesPedigreeFunction_0.6 이상', 'Age_50세 이상']


,Pregnancies_0회,Pregnancies_1-2회,Pregnancies_3-5회,Pregnancies_6회 이상,Glucose_100 미만,Glucose_100-139,Glucose_140-169,Glucose_170 이상,Glucose_미측정,BloodPressure_70 미만,...,BMI_30-34.9,BMI_35 이상,BMI_미측정,DiabetesPedigreeFunction_0.3 미만,DiabetesPedigreeFunction_0.3-0.59,DiabetesPedigreeFunction_0.6 이상,Age_20대,Age_30대,Age_40대,Age_50세 이상
0,0,0,0,1,0,0,1,0,0,0,...,1,0,0,0,0,1,0,0,0,1
1,0,1,0,0,1,0,0,0,0,1,...,0,0,0,0,1,0,0,1,0,0
2,0,0,0,1,0,0,0,1,0,1,...,0,0,0,0,0,1,0,1,0,0


## [2단계] 데이터셋 분할 (학습용 / 테스트용)
원래 실습과 결과를 비교할 수 있도록 `test_size=0.2`, `random_state=42`를 그대로 씁니다. 행이 똑같이 나뉘므로 원본과 원핫 결과를 같은 환자로 비교하게 됩니다.

In [7]:
# [2단계] 데이터셋 분할 (학습용 / 테스트용)
from sklearn.model_selection import train_test_split

# 전체 데이터 중 80%는 학습용, 20%는 테스트용으로 분할 (재현성을 위해 random_state 설정)
X_train, X_test, y_train, y_test = train_test_split(X_onehot, y, test_size=0.2, random_state=42)

print(f"학습용 데이터 크기: X_train {X_train.shape}, y_train {y_train.shape}")
print(f"테스트용 데이터 크기: X_test {X_test.shape}, y_test {y_test.shape}")

학습용 데이터 크기: X_train (614, 35), y_train (614,)
테스트용 데이터 크기: X_test (154, 35), y_test (154,)


## 모델 1. 의사결정나무

In [8]:
# [3단계] 모델 객체 생성
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(random_state=42)

# [4단계] 모델 학습
model.fit(X_train, y_train)
print("의사결정나무 모델 학습이 완료되었습니다.")

# [5단계] 예측 및 성능 평가
from sklearn.metrics import classification_report
pred_dt = model.predict(X_test)
print("\n[의사결정나무 분류 성능 평가 리포트 - 원핫인코딩]")
print(classification_report(y_test, pred_dt))

의사결정나무 모델 학습이 완료되었습니다.

[의사결정나무 분류 성능 평가 리포트 - 원핫인코딩]
              precision    recall  f1-score   support

           0       0.78      0.77      0.78        99
           1       0.60      0.62      0.61        55

    accuracy                           0.71       154
   macro avg       0.69      0.69      0.69       154
weighted avg       0.72      0.71      0.72       154



## 모델 2. 랜덤포레스트

In [9]:
# [3단계] 모델 객체 생성 (랜덤 포레스트)
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(random_state=42)

# [4단계] 모델 학습
model.fit(X_train, y_train)
print("랜덤 포레스트 모델 학습이 완료되었습니다.")

# [5단계] 예측 및 성능 평가
pred_rf = model.predict(X_test)
print("\n[랜덤 포레스트 분류 성능 평가 리포트 - 원핫인코딩]")
print(classification_report(y_test, pred_rf))

랜덤 포레스트 모델 학습이 완료되었습니다.

[랜덤 포레스트 분류 성능 평가 리포트 - 원핫인코딩]
              precision    recall  f1-score   support

           0       0.76      0.81      0.78        99
           1       0.61      0.55      0.58        55

    accuracy                           0.71       154
   macro avg       0.69      0.68      0.68       154
weighted avg       0.71      0.71      0.71       154



## 모델 3. 로지스틱 회귀

In [10]:
# [3단계] 모델 객체 생성 (로지스틱 회귀)
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=200, random_state=42)

# [4단계] 모델 학습
model.fit(X_train, y_train)
print("로지스틱 회귀 모델 학습이 완료되었습니다.")

# [5단계] 예측 및 성능 평가
pred_lr = model.predict(X_test)
print("\n[로지스틱 회귀 분류 성능 평가 리포트 - 원핫인코딩]")
print(classification_report(y_test, pred_lr))

lr_model = model    # 8단계에서 계수를 해석할 때 씁니다

로지스틱 회귀 모델 학습이 완료되었습니다.

[로지스틱 회귀 분류 성능 평가 리포트 - 원핫인코딩]
              precision    recall  f1-score   support

           0       0.82      0.79      0.80        99
           1       0.64      0.69      0.67        55

    accuracy                           0.75       154
   macro avg       0.73      0.74      0.74       154
weighted avg       0.76      0.75      0.76       154



## 모델 4. KNN
KNN은 거리로 가까운 이웃을 찾습니다. 원본에서는 Insulin(0~846)처럼 값의 범위가 큰 column이 거리를 좌우했지만, 원핫인코딩 후에는 모든 column이 0과 1이라 범위가 같아집니다.

In [11]:
# [3단계] 모델 객체 생성 (KNN)
from sklearn.neighbors import KNeighborsClassifier
model = KNeighborsClassifier()

# [4단계] 모델 학습 (학습 데이터를 저장해 둡니다)
model.fit(X_train, y_train)
print("KNN 모델 학습이 완료되었습니다.")

# [5단계] 예측 및 성능 평가
pred_knn = model.predict(X_test)
print("\n[KNN 분류 성능 평가 리포트 - 원핫인코딩]")
print(classification_report(y_test, pred_knn))

KNN 모델 학습이 완료되었습니다.

[KNN 분류 성능 평가 리포트 - 원핫인코딩]
              precision    recall  f1-score   support

           0       0.75      0.86      0.80        99
           1       0.66      0.49      0.56        55

    accuracy                           0.73       154
   macro avg       0.71      0.67      0.68       154
weighted avg       0.72      0.73      0.72       154



## 모델 5. 의사결정나무 회귀
원래 실습처럼 0과 1을 숫자로 보고 회귀로 예측한 뒤 RMSE를 구합니다. RMSE는 작을수록 좋습니다.

In [12]:
# [3단계] 회귀 모델 객체 생성
from sklearn.tree import DecisionTreeRegressor
model = DecisionTreeRegressor(random_state=42)

# [4단계] 모델 학습
model.fit(X_train, y_train)
print("의사결정나무 회귀 모델 학습이 완료되었습니다.")

# [5단계] 예측 및 회귀 성능 평가
from sklearn.metrics import mean_squared_error
dy_preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, dy_preds))
print("평균제곱근오차(RMSE) - 원핫인코딩:", rmse)

의사결정나무 회귀 모델 학습이 완료되었습니다.
평균제곱근오차(RMSE) - 원핫인코딩: 0.5432253630803656


## [6단계] 원래 실습(원본 숫자)과 비교
같은 모델, 같은 분할로 원본 숫자 데이터와 원핫인코딩 데이터를 나란히 비교합니다. 당뇨 재현율은 실제 당뇨 환자(1) 중 모델이 당뇨로 맞힌 비율입니다.

In [13]:
# [6단계] 원래 실습(원본 숫자)과 비교
from sklearn.metrics import accuracy_score, recall_score, f1_score

def make_models():
    return {
        "의사결정나무": DecisionTreeClassifier(random_state=42),
        "랜덤포레스트": RandomForestClassifier(random_state=42),
        "로지스틱 회귀": LogisticRegression(max_iter=1000, random_state=42),
        "KNN": KNeighborsClassifier(),
    }

datasets = {"원본": X, "원핫": X_onehot}   # 원본 = ex_0914에서 쓴 데이터 그대로
rows = []
for data_name, data in datasets.items():
    Xtr, Xte, ytr, yte = train_test_split(data, y, test_size=0.2, random_state=42)
    for model_name, m in make_models().items():
        pred = m.fit(Xtr, ytr).predict(Xte)
        rows.append({"모델": model_name, "데이터": data_name,
                     "정확도": accuracy_score(yte, pred),
                     "당뇨 재현율": recall_score(yte, pred),
                     "당뇨 F1": f1_score(yte, pred)})
    reg = DecisionTreeRegressor(random_state=42).fit(Xtr, ytr)
    rows.append({"모델": "의사결정나무 회귀 (RMSE)", "데이터": data_name,
                 "정확도": np.nan, "당뇨 재현율": np.nan, "당뇨 F1": np.nan,
                 "RMSE": np.sqrt(mean_squared_error(yte, reg.predict(Xte)))})

compare = pd.DataFrame(rows).pivot(index="모델", columns="데이터")
compare = compare.reindex(["의사결정나무", "랜덤포레스트", "로지스틱 회귀", "KNN", "의사결정나무 회귀 (RMSE)"])
display(compare.round(3))

정확도        당뇨 재현율         당뇨 F1          RMSE       
데이터                  원본     원핫     원본     원핫     원본     원핫     원본     원핫
모델                                                                      
의사결정나무            0.747  0.714  0.727  0.618  0.672  0.607    NaN    NaN
랜덤포레스트            0.721  0.714  0.618  0.545  0.613  0.577    NaN    NaN
로지스틱 회귀           0.747  0.753  0.673  0.691  0.655  0.667    NaN    NaN
KNN               0.662  0.727  0.582  0.491  0.552  0.562    NaN    NaN
의사결정나무 회귀 (RMSE)    NaN    NaN    NaN    NaN    NaN    NaN  0.503  0.543

## [7단계] 교차검증으로 다시 비교
테스트 데이터는 154명뿐이라 몇 명만 다르게 맞혀도 정확도가 1~2%씩 움직입니다. 그래서 데이터를 5번 다르게 나눠 평균을 내는 교차검증으로 한 번 더 비교합니다.

In [14]:
# [7단계] 5-fold 교차검증 정확도 비교
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_rows = []
for model_name in make_models():
    row = {"모델": model_name}
    for data_name, data in datasets.items():
        scores = cross_val_score(make_models()[model_name], data, y, cv=cv, scoring="accuracy")
        row[f"{data_name} 평균"] = scores.mean()
        row[f"{data_name} 표준편차"] = scores.std()
    cv_rows.append(row)

display(pd.DataFrame(cv_rows).set_index("모델").round(3))
print("참고: 전부 0(당뇨 아님)으로 찍어도 정확도는", round((y == 0).mean(), 3), "입니다.")

,원본 평균,원본 표준편차,원핫 평균,원핫 표준편차
모델,,,,
의사결정나무,0.700,0.041,0.671,0.027
랜덤포레스트,0.769,0.018,0.747,0.031
로지스틱 회귀,0.771,0.015,0.764,0.024
KNN,0.698,0.033,0.693,0.019


참고: 전부 0(당뇨 아님)으로 찍어도 정확도는 0.651 입니다.


## [8단계] 로지스틱 회귀 계수로 구간 해석하기
원핫인코딩의 장점은 **구간마다** 영향을 읽을 수 있다는 것입니다. 계수가 양수면 그 구간에 속할 때 당뇨 쪽으로, 음수면 정상 쪽으로 예측이 기웁니다.

In [15]:
# [8단계] 로지스틱 회귀 계수 해석
coef = pd.Series(lr_model.coef_[0], index=X_train.columns).sort_values()

print("당뇨 쪽으로 가장 크게 기우는 구간 TOP 5")
display(coef.tail(5)[::-1].round(2).to_frame("계수"))

print("정상 쪽으로 가장 크게 기우는 구간 TOP 5")
display(coef.head(5).round(2).to_frame("계수"))

당뇨 쪽으로 가장 크게 기우는 구간 TOP 5


,계수
Glucose_170 이상,1.74
BMI_35 이상,0.88
BMI_30-34.9,0.82
BloodPressure_미측정,0.62
Age_50세 이상,0.59


정상 쪽으로 가장 크게 기우는 구간 TOP 5


,계수
Glucose_100 미만,-1.91
BMI_25 미만,-1.19
Age_20대,-0.91
DiabetesPedigreeFunction_0.3 미만,-0.53
BloodPressure_80-89,-0.52


## 정리

- **원핫인코딩은 범주형 데이터용입니다.** 숫자만 있는 데이터에 쓰려면 먼저 구간으로 나눠 범주로 만들어야 합니다.
- **0을 미측정 구간으로 따로 둘 수 있습니다.** 원핫인코딩에서는 빠진 값도 하나의 구간이 되므로, 평균 같은 값으로 채우지 않아도 됩니다.
- **정확도가 꼭 오르지는 않습니다.** 구간으로 나누면 같은 구간 안의 값 차이(예: 혈당 141과 169)가 사라집니다. 특히 나무 모델은 원래 숫자에서 스스로 경계를 찾으므로 정보가 줄어든 만큼 손해를 볼 수 있습니다.
- **대신 해석이 쉬워집니다.** "혈당 170 이상", "BMI 35 이상" 같은 구간 단위로 영향을 설명할 수 있습니다.
- **KNN은 값의 범위에 민감합니다.** 원핫인코딩으로 모든 column이 0과 1이 되면 범위 차이 문제가 사라집니다.

## 더 해볼 것

1. `BINS`의 구간 경계를 바꿔 보고 결과가 어떻게 달라지는지 비교하기
2. `pd.get_dummies(X_cat, dtype=int, drop_first=True)`로 column마다 첫 구간을 빼고 실행해 보기
3. 로지스틱 회귀에 `class_weight="balanced"`를 넣어 당뇨 재현율이 얼마나 오르는지 확인하기
4. 원본 숫자 데이터에 `StandardScaler`를 적용한 KNN과 원핫 KNN을 비교하기